# Supplementary Figure S9 (H-I)

Depth preference across visual space without the spherical-viewing-angle correction.
Panels A-G come from `figsupp9_v1_depth_map.ipynb`. The pairwise RF distance panels
that used to be S9 A-C are now Figure 4 I-K (`figure4_receptive_fields.ipynb`).

In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import spheres, common_utils, roi_location
from cottage_analysis.plotting import rf_plots
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.summary_analysis import get_session_list
from cottage_analysis.plotting import style

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
import os

# RF fits used by every RF panel: "neurons_df_spks.pickle" (RFs refitted on deconvolved
# spikes, used in the manuscript; other columns stay dF/F) or "neurons_df.pickle" (dF/F).
# Outputs and caches get a suffix.
RF_FILE = "neurons_df_spks.pickle"
OUT_SFX = "" if RF_FILE == "neurons_df.pickle" else "_" + RF_FILE[len("neurons_df_"):-len(".pickle")]
print(f"RF fits from {RF_FILE}, output suffix {OUT_SFX!r}")

project = "hey2_3d-vision_foodres_20220101"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
(SAVE_ROOT / "rf_supp").mkdir(parents=True, exist_ok=True)

## Supplementary Figure S9 (H-I): depth preference across visual space - UNCORRECTED depth

Same panels as Figure S9E-G, but built on `preferred_depth_closedloop` instead of
`preferred_depth_corrected`, i.e. without the spherical-viewing-angle correction.


In [ ]:
# Load the neuron table used by figsupp9_v1_depth_map.ipynb and reproduce its neuron selection.
import pickle
import matplotlib.path as mplPath
from roifile import ImagejRoi
from cottage_analysis.plotting.style import rect_cm, panel_letter
from v1_depth_map.figure_utils import v1_map

# caches written by figsupp9_v1_depth_map (with the same RF_FILE)
neurons_df_all = pd.read_pickle(READ_ROOT / f"rf_neurons_df_slim{OUT_SFX}.pkl")

data_root = flz.get_data_root("processed", flexilims_session=flexilims_session)
overview_fname = data_root / project / "PZAH10.2d/FOVs/PZAH10.2d_overview.tif"
v1_mask_fname = data_root / project / "PZAH10.2d/FOVs/V1_mask_2.roi"
overview_img = plt.imread(overview_fname)

# Per-neuron V1 membership (see figsupp9_v1_depth_map.ipynb): finer than the per-session `v1` flag
_x = neurons_df_all["overview_x_aligned"].to_numpy(float)
_y = neurons_df_all["overview_y_aligned"].to_numpy(float)
_h, _w = overview_img.shape[:2]
_inside_img = (
    np.isfinite(_x) & np.isfinite(_y) & (_x >= 0) & (_x < _w) & (_y >= 0) & (_y < _h)
)
neurons_df_all["v1_mask"] = False
neurons_df_all.loc[_inside_img, "v1_mask"] = mplPath.Path(
    ImagejRoi.fromfile(v1_mask_fname).coordinates()
).contains_points(np.column_stack([_x[_inside_img], _y[_inside_img]]))

common_utils.add_one_sided_spearman_significance(
    neurons_df_all,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="is_depth_neuron",
)

use_v1_mask = True  # keep in sync with figsupp9_v1_depth_map.ipynb
sig_neurons = (
    (neurons_df_all["iscell"] == True)
    & (neurons_df_all["is_depth_neuron"])
    & (neurons_df_all["v1"] == True)
    & (neurons_df_all["rf_sig"] == True)
)
if use_v1_mask:
    sig_neurons = sig_neurons & neurons_df_all["v1_mask"]

# Jitter visual space coordinates for scatter plotting (same seed as figsupp9_v1_depth_map.ipynb)
np.random.seed(0)
neurons_df_all["rf_azi_jittered"] = neurons_df_all["rf_azi"] + np.random.uniform(
    -2, 2, neurons_df_all.shape[0]
)
neurons_df_all["rf_ele_jittered"] = neurons_df_all["rf_ele"] + np.random.uniform(
    -2, 2, neurons_df_all.shape[0]
)
neurons_df_sig = neurons_df_all[sig_neurons]

# Uncorrected gradient bootstrap ("coefs"); "coefs_corrected" is the Figure 5 version
with open(READ_ROOT / f"rf_gradient_bootstraps{OUT_SFX}.pkl", "rb") as f:
    rf_gradient_bootstraps = pickle.load(f)
coefs_uncorrected = np.array(rf_gradient_bootstraps["coefs"])

print(f"{len(neurons_df_sig)} depth-selective neurons with significant RFs")

In [ ]:
# Bin uncorrected preferred depth over visual space
DEPTH_COL = (
    "preferred_depth_closedloop"  # uncorrected; Figure 5 uses preferred_depth_corrected
)
MIN_NEURONS_PER_BIN = 20

xrange_vs = 17.5, 117.5
yrange_vs = -37.5, 37.5
xx_vs, yy_vs = np.mgrid[
    xrange_vs[0] : xrange_vs[1] + 5 : 5, yrange_vs[0] : yrange_vs[1] + 5 : 5
]
im_extent = (
    xrange_vs[0] - 2.5,
    xrange_vs[1] + 2.5,
    yrange_vs[0] - 2.5,
    yrange_vs[1] + 2.5,
)

median_depth = np.ravel(np.zeros(xx_vs.shape))
for i, (x, y) in enumerate(zip(np.ravel(xx_vs), np.ravel(yy_vs))):
    idx = (neurons_df_sig["rf_azi"] == x) & (neurons_df_sig["rf_ele"] == y)
    if np.sum(idx) < MIN_NEURONS_PER_BIN:
        median_depth[i] = np.nan
    else:
        median_depth[i] = np.median(neurons_df_sig[idx][DEPTH_COL])
median_depth = np.reshape(median_depth, xx_vs.shape).T


def calculate_zz(condition):
    zz = np.ravel(np.zeros(xx_vs.shape))
    for i, (x, y) in enumerate(zip(np.ravel(xx_vs), np.ravel(yy_vs))):
        idx = (neurons_df_sig["rf_azi"] == x) & (neurons_df_sig["rf_ele"] == y)
        if np.sum(idx) < MIN_NEURONS_PER_BIN:
            zz[i] = np.nan
        else:
            zz[i] = np.mean(condition[idx])
    return np.reshape(zz, xx_vs.shape).T

In [ ]:
# Draw the two panels on their own canvas, reusing the Figure 5E/F geometry
FIG_W, FIG_H = v1_map.FIG_W, 5.30  # cm; same width as Figure 5, only the E/F band tall
fig = plt.figure(figsize=(FIG_W / CM, FIG_H / CM), dpi=300)
fig.patch.set_facecolor("white")

panel_letter(fig, "H", 0.15, 5.00)
panel_letter(fig, "I", 7.80, 5.00)

v1_map.plot_panel_e(
    fig,
    rect_cm(fig, *v1_map.LAYOUT["panel_e_main"]),
    rect_cm(fig, *v1_map.LAYOUT["panel_e_polar"]),
    rect_cm(fig, *v1_map.LAYOUT["panel_e_cbar"]),
    median_depth,
    coefs_uncorrected,
    im_extent,
)

v1_map.plot_panel_f(
    fig,
    [rect_cm(fig, *r) for r in v1_map.LAYOUT["panel_f_scatter"]],
    [rect_cm(fig, *r) for r in v1_map.LAYOUT["panel_f_prop"]],
    rect_cm(fig, *v1_map.LAYOUT["panel_f_cbar"]),
    neurons_df_sig,
    im_extent,
    calculate_zz,
    plt.Normalize(vmin=np.log(0.05), vmax=np.log(6.4)),
    depth_col=DEPTH_COL,
)

save_path = SAVE_ROOT / "rf_supp" / f"v1map_uncorrected{OUT_SFX}.svg"
style.savefig(save_path, fig=fig, dpi=300, bbox_inches="tight")
print(f"Saved to: {save_path}")
plt.show()

In [ ]:
# Uncorrected vs corrected gradient direction, for the legend
from cottage_analysis.analysis.common_utils import ztest_2d

coefs_corrected = np.array(rf_gradient_bootstraps["coefs_corrected"])
for name, c in [("uncorrected", coefs_uncorrected), ("corrected", coefs_corrected)]:
    angles = np.rad2deg(np.arctan2(c[:, 1], c[:, 0]))
    print(
        f"{name:>12}: p = {ztest_2d(c)[0]:.3e}, "
        f"gradient direction = {np.median(angles):.1f} deg "
        f"[{np.quantile(angles, 0.025):.1f}, {np.quantile(angles, 0.975):.1f}]"
    )